In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('etl') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [4]:
from pyspark.sql.types import StructType, StructField, StringType

## Orders Events

In [5]:
orders_path = '../Data/order_events.jsonl'

order_schema = StructType([
    StructField("event_id", StringType(),  False),
    StructField("user_id", StringType(),  False),
    StructField("order_id", StringType(),  False),  
    StructField("product_id", StringType(),  False),
    StructField("event_type", StringType(),  False),  
    StructField("price", StringType(),  False),
    StructField("region", StringType(),  False),
    StructField("event_timestamp", StringType(),  False),
    StructField("is_delayed", StringType(),  False),
    StructField("is_returned", StringType(),  False),
])


order_df = (
    spark.read    
    .schema(order_schema)             
    .json(orders_path)
)

In [6]:
order_df.show(5)

+--------------------+-------+------------+----------+-----------------+-------+------+--------------------+----------+-----------+
|            event_id|user_id|    order_id|product_id|       event_type|  price|region|     event_timestamp|is_delayed|is_returned|
+--------------------+-------+------------+----------+-----------------+-------+------+--------------------+----------+-----------+
|20b92918-abdd-454...|   7231|ORD-d4183470|   PRD-098|    ORDER_CREATED|1703.07|    BA|2025-06-13T00:59:...|      NULL|       NULL|
|429dc28f-00e4-467...|   7231|ORD-d4183470|   PRD-098|PAYMENT_CONFIRMED|1703.07|    BA|2025-06-13T01:04:...|      NULL|       NULL|
|560a39fe-6289-4e6...|   7231|ORD-d4183470|   PRD-098|     ORDER_PACKED|1703.07|    BA|2025-06-13T01:59:...|      NULL|       NULL|
|0c71b51e-cd54-48c...|   7231|ORD-d4183470|   PRD-098|    ORDER_SHIPPED|1703.07|    BA|2025-06-13T02:59:...|      NULL|       NULL|
|5affd54a-1522-407...|   7231|ORD-d4183470|   PRD-098|  ORDER_DELIVERED|1703

In [7]:
(
    order_df
    .writeTo("iceberg.bronze.tbl_bronze_order_events")
    .createOrReplace()
)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Products Catalog

In [ ]:
product_catalog_path = '../Data/products.csv'

product_catalog_schema = StructType([
    StructField("product_id", StringType(),  False),
    StructField("product_name", StringType(),  False),
    StructField("category", StringType(),  False),
    StructField("price", StringType(),  False),
])


product_catalog_df = (
    spark.read
    .option("header", True)    
    .option("delimiter", ",")   
    .schema(product_catalog_schema)             
    .csv(product_catalog_path)
)


In [ ]:
product_catalog_df.show(5)

In [ ]:
(
    product_catalog_df
    .writeTo("iceberg.bronze.tbl_bronze_product_catalog")
    .createOrReplace()
)

In [2]:
spark.stop()